# derived_8.3-eval-1.0 — Parallel MoE Evaluation under SOTA 1.5 Hyperparameters

This experiment evaluates the **2-regime Mixture-of-Experts (MoE) models** and baseline models on the Washington-only `derived_8.3` split, utilizing the **SOTA 1.5 hyperparameters** optimized during the `derived_8.2-hyperparameters-1.5` sweep. We replace `derived_8.2`'s `OVERALL_SELECTED_FEATURES_V3` with `derived_8.3`'s `OVERALL_SELECTED_FEATURES_V0` (50 features loaded from `data/splits/derived_8.3/dataset_metadata.py`). We evaluate 5 gating strategies ($K=2$): `Trained_Gating`, `Univariate_G_API`, `Clustering_Dynamic`, `Seasonal_Binary`, and `Clustering_V0_Full`.

# Section 1: Setup and configuration

Import libraries, fix random seeds for reproducibility, locate project root, probe CUDA availability for XGBoost acceleration, and configure parallel training workers (`XGB_PARALLEL_WORKERS`, default 4). Output artifacts land under this experiment directory.

In [1]:
import os
import sys
import random
import time
import json
import zlib
import threading
import importlib.util
from concurrent.futures import ThreadPoolExecutor, as_completed
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.cm as cm
from sklearn.metrics import (
    mean_absolute_error,
    r2_score,
    root_mean_squared_error,
    accuracy_score,
    precision_recall_fscore_support,
    confusion_matrix,
)
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler
import xgboost as xgb
from xgboost import XGBRegressor, XGBClassifier


def find_project_root():
    candidates = [Path.cwd().resolve(), *Path.cwd().resolve().parents]
    for cand in candidates:
        if (cand / "data").exists() and (cand / "d_models").exists():
            return cand
    raise FileNotFoundError("Could not locate repo root containing 'data' and 'd_models'")


PROJECT_ROOT = find_project_root()
EXP_DIR = PROJECT_ROOT / "notebooks/experiment/derived_8.3-eval-1.0"
out_dir = EXP_DIR
out_dir.mkdir(parents=True, exist_ok=True)

SEED = 42
random.seed(SEED)
np.random.seed(SEED)

PARALLEL_WORKERS = int(os.getenv("XGB_PARALLEL_WORKERS", "4"))
print(f"Project root: {PROJECT_ROOT}")
print(f"Output dir: {out_dir}")
print(f"XGBoost version: {xgb.__version__}")
print(f"Parallel workers: {PARALLEL_WORKERS}")

# SOTA 1.5 Hyperparameters
XGB_REG_PARAMS = {
    "n_estimators": 2500,
    "learning_rate": 0.005,
    "max_depth": 9,
    "min_child_weight": 8,
    "gamma": 0.0,
    "reg_lambda": 0.75,
    "reg_alpha": 0.03,
    "subsample": 0.9,
    "colsample_bytree": 0.8,
    "random_state": SEED,
    "tree_method": "hist",
    "device": "cuda",
    "n_jobs": 1,
}

XGB_CLF_PARAMS = {
    "n_estimators": 500,
    "learning_rate": 0.01,
    "max_depth": 6,
    "subsample": 0.8,
    "colsample_bytree": 0.8,
    "random_state": SEED,
    "tree_method": "hist",
    "device": "cuda",
    "n_jobs": 1,
    "eval_metric": "logloss",
}

Project root: /scratch/user/u.rp352032/MDR-Project
Output dir: /scratch/user/u.rp352032/MDR-Project/notebooks/experiment/derived_8.3-eval-1.0
XGBoost version: 3.2.0
Parallel workers: 4


# Section 2: Load data splits

Load `train.csv`, `val.csv`, and `test.csv` from `data/splits/derived_8.3/` (9 clean Washington SNOTEL and baseline stations). Extract temporal features (`month`, `year`) and construct the concatenated `trainval_df` training set.

In [2]:
TRAIN_PATH = PROJECT_ROOT / "data/splits/derived_8.3/train.csv"
VAL_PATH = PROJECT_ROOT / "data/splits/derived_8.3/val.csv"
TEST_PATH = PROJECT_ROOT / "data/splits/derived_8.3/test.csv"
METADATA_PATH = PROJECT_ROOT / "data/splits/derived_8.3/dataset_metadata.py"
TARGET_COL = "soil_moisture_5cm"
T_BINARY = 0.16

train_df = pd.read_csv(TRAIN_PATH)
val_df = pd.read_csv(VAL_PATH)
test_df = pd.read_csv(TEST_PATH)

print("Dataset splits loaded:")
print(f"  train: {train_df.shape}")
print(f"  val:   {val_df.shape}")
print(f"  test:  {test_df.shape}")

for df in [train_df, val_df, test_df]:
    df["date"] = pd.to_datetime(df["date"])
    df["month"] = df["date"].dt.month.astype(int)
    df["year"] = df["date"].dt.year.astype(float)

trainval_df = pd.concat([train_df, val_df], axis=0).reset_index(drop=True)
print(f"  trainval (concatenated): {trainval_df.shape}")

y_trval = trainval_df[TARGET_COL].values
y_te = test_df[TARGET_COL].values
y_train_only = train_df[TARGET_COL].values
test_years = sorted(test_df["year"].unique())
print(f"Test years: {test_years}")

spec = importlib.util.spec_from_file_location("dataset_metadata", METADATA_PATH)
dm = importlib.util.module_from_spec(spec)
spec.loader.exec_module(dm)
OVERALL_SELECTED_FEATURES_V0 = list(dm.OVERALL_SELECTED_FEATURES_V0)
print(f"Loaded OVERALL_SELECTED_FEATURES_V0: {len(OVERALL_SELECTED_FEATURES_V0)} features")

Dataset splits loaded:
  train: (12678, 499)
  val:   (6219, 499)
  test:  (8396, 499)
  trainval (concatenated): (18897, 500)
Test years: [np.float64(2023.0), np.float64(2024.0), np.float64(2025.0)]
Loaded OVERALL_SELECTED_FEATURES_V0: 50 features


/tmp/job.1971555/ipykernel_3133442/2087608658.py:19: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df["month"] = df["date"].dt.month.astype(int)
/tmp/job.1971555/ipykernel_3133442/2087608658.py:19: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df["month"] = df["date"].dt.month.astype(int)
/tmp/job.1971555/ipykernel_3133442/2087608658.py:19: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using p

# Section 3: Feature inventory

Load feature lists from `previous_features.json` and `selected_features.json`. Verify that all referenced features are present in the dataset columns.

In [3]:
prev_path = out_dir / "previous_features.json"
new_path = out_dir / "selected_features.json"

if not prev_path.exists():
    raise FileNotFoundError(f"Missing {prev_path}")
if not new_path.exists():
    raise FileNotFoundError(
        f"Missing {new_path}. Run run_feature_selection.py first."
    )

with open(prev_path, "r") as f:
    PREV = json.load(f)
with open(new_path, "r") as f:
    NEW = json.load(f)

if NEW.get("partial"):
    print("[WARNING] selected_features.json is marked partial — selection may be incomplete.")

FEATURE_SET_V0 = list(OVERALL_SELECTED_FEATURES_V0)
FEATURE_SET_C1 = list(NEW.get("global_c1", {}).get("features", OVERALL_SELECTED_FEATURES_V0))

REGIME_OLD = {
    "dry": list(PREV.get("binary_regime", {}).get("dry", {}).get("features", OVERALL_SELECTED_FEATURES_V0)),
    "wet": list(PREV.get("binary_regime", {}).get("wet", {}).get("features", OVERALL_SELECTED_FEATURES_V0)),
}
REGIME_NEW = {
    "dry": list(NEW.get("binary_regime", {}).get("dry", {}).get("features", OVERALL_SELECTED_FEATURES_V0)),
    "wet": list(NEW.get("binary_regime", {}).get("wet", {}).get("features", OVERALL_SELECTED_FEATURES_V0)),
}

CLUSTER_OLD = {
    strat: {c: list(payload["features"]) for c, payload in clusters.items()}
    for strat, clusters in PREV.get("clusters", {}).items()
}
CLUSTER_NEW = {
    strat: {c: list(payload["features"]) for c, payload in clusters.items()}
    for strat, clusters in NEW.get("clusters", {}).items()
}

print(f"Global V0: {len(FEATURE_SET_V0)} features")
print(f"Global c1: {len(FEATURE_SET_C1)} features")
print(f"Binary new dry/wet: {len(REGIME_NEW['dry'])}/{len(REGIME_NEW['wet'])}")
for strat in CLUSTER_NEW:
    print(f"  Cluster new {strat}: " + ", ".join(f"c{c}={len(feats)}" for c, feats in CLUSTER_NEW[strat].items()))

all_needed = set(FEATURE_SET_V0) | set(FEATURE_SET_C1)
for feats in REGIME_NEW.values():
    all_needed |= set(feats)
for strat in CLUSTER_NEW.values():
    for feats in strat.values():
        all_needed |= set(feats)

missing = sorted(c for c in all_needed if c not in trainval_df.columns)
if missing:
    raise ValueError(f"{len(missing)} features missing from trainval: {missing[:20]}")
print(f"All {len(all_needed)} referenced features present in trainval.")

Global V0: 50 features
Global c1: 50 features
Binary new dry/wet: 28/50
  Cluster new Univariate_G_API_k2: c0=45, c1=50
  Cluster new Clustering_Dynamic_k2: c0=1, c1=20
  Cluster new Seasonal_Binary_k2: c0=15, c1=1
  Cluster new Clustering_V0_Full_k2: c0=4, c1=47
All 165 referenced features present in trainval.


# Section 4: Helpers (metrics, routing, plots)

Define evaluation metric functions (R2, RMSE, ubRMSE, Bias, MAE, Med|Err|, Pearson), quantile binning, KMeans clustering, regime routing logic, and visualization utilities for diagnostic loss curves and per-regime residual analyses.

In [4]:
def compute_metrics(y_true, y_pred):
    y_true = np.asarray(y_true, dtype=float).ravel()
    y_pred = np.asarray(y_pred, dtype=float).ravel()
    mask = ~(np.isnan(y_true) | np.isnan(y_pred))
    y_true, y_pred = y_true[mask], y_pred[mask]
    if len(y_true) < 2:
        return {
            "R2": float("nan"),
            "RMSE": float("nan"),
            "ubRMSE": float("nan"),
            "Bias": float("nan"),
            "MAE": float("nan"),
            "Med|Err|": float("nan"),
            "Pearson": float("nan"),
        }

    r2 = float(r2_score(y_true, y_pred))
    rmse = float(root_mean_squared_error(y_true, y_pred))
    bias = float(np.mean(y_pred - y_true))
    ubrmse = float(np.sqrt(max(0.0, rmse**2 - bias**2)))
    mae = float(mean_absolute_error(y_true, y_pred))
    med_abs = float(np.median(np.abs(y_pred - y_true)))

    std_t = np.std(y_true)
    std_p = np.std(y_pred)
    if std_t > 1e-12 and std_p > 1e-12:
        pearson = float(np.corrcoef(y_true, y_pred)[0, 1])
    else:
        pearson = float("nan")

    return {
        "R2": r2,
        "RMSE": rmse,
        "ubRMSE": ubrmse,
        "Bias": bias,
        "MAE": mae,
        "Med|Err|": med_abs,
        "Pearson": pearson,
    }


class QuantileBinner:
    def __init__(self, K: int):
        self.K = K
        self.thresholds: list[float] = []

    def fit(self, series: pd.Series):
        val = series.fillna(series.mean())
        self.thresholds = [float(val.quantile(i / self.K)) for i in range(1, self.K)]

    def predict(self, series: pd.Series) -> np.ndarray:
        val = series.fillna(series.mean())
        if self.K == 2:
            return np.where(val < self.thresholds[0], 0, 1)
        raise NotImplementedError("Only K=2 supported in eval-1.0")


class KMeansClusterer:
    def __init__(self, cols: list[str], K: int):
        self.cols = cols
        self.K = K
        self.means = None
        self.scaler = StandardScaler()
        self.kmeans = KMeans(n_clusters=K, random_state=SEED, n_init=10)

    def fit(self, df: pd.DataFrame):
        X = df[self.cols].copy()
        self.means = X.mean()
        X = X.fillna(self.means)
        self.kmeans.fit(self.scaler.fit_transform(X))

    def predict(self, df: pd.DataFrame) -> np.ndarray:
        X = df[self.cols].copy().fillna(self.means)
        return self.kmeans.predict(self.scaler.transform(X))


COLS_DYNAMIC = ["SMAP_sm_pm_interp_lag1", "G_API", "LST_modis"]


def plot_diagnostics(mname, y_true, y_pred, df, out_dir):
    fig, axes = plt.subplots(1, 2, figsize=(12, 5))
    ax = axes[0]
    ax.scatter(y_true, y_pred, alpha=0.2, s=8, color="#1f77b4")
    m_val = max(np.max(y_true), np.max(y_pred))
    ax.plot([0, m_val], [0, m_val], "r--", linewidth=1.5)
    ax.set_xlabel("Observed SM")
    ax.set_ylabel("Predicted SM")
    ax.set_title(f"Scatter: {mname}")
    ax.grid(True, linestyle="--", alpha=0.5)

    ax = axes[1]
    res = y_pred - y_true
    ax.hist(res, bins=50, color="#2ca02c", alpha=0.7, edgecolor="k")
    ax.axvline(0, color="r", linestyle="--")
    ax.set_xlabel("Residual (Pred - Obs)")
    ax.set_ylabel("Count")
    ax.set_title(f"Residuals: {mname}")
    ax.grid(True, linestyle="--", alpha=0.5)

    plt.tight_layout()
    sanitized = mname.replace(":", "").replace(" ", "_").replace("=", "").replace("/", "_")
    plt.savefig(out_dir / f"diag_{sanitized}.png", dpi=120)
    plt.close()


def plot_per_regime_diagnostics(mname, y_true, y_pred, labels, df, out_dir):
    fig, axes = plt.subplots(1, 2, figsize=(12, 5))
    colors = ["#1f77b4", "#ff7f0e"]
    for c in range(2):
        mask = labels == c
        if not mask.any():
            continue
        ax = axes[0]
        ax.scatter(
            y_true[mask],
            y_pred[mask],
            alpha=0.3,
            s=10,
            color=colors[c],
            label=f"Regime {c}",
        )

        ax = axes[1]
        res = (y_pred - y_true)[mask]
        ax.hist(res, bins=40, alpha=0.5, color=colors[c], label=f"Regime {c}", density=True)

    axes[0].plot([0, 0.6], [0, 0.6], "k--", alpha=0.7)
    axes[0].set_xlabel("Observed")
    axes[0].set_ylabel("Predicted")
    axes[0].set_title(f"Per-Regime Scatter: {mname}")
    axes[0].legend()
    axes[0].grid(True, linestyle="--", alpha=0.5)

    axes[1].axvline(0, color="k", linestyle="--")
    axes[1].set_xlabel("Residual")
    axes[1].set_ylabel("Density")
    axes[1].set_title(f"Per-Regime Residual Density: {mname}")
    axes[1].legend()
    axes[1].grid(True, linestyle="--", alpha=0.5)

    plt.tight_layout()
    sanitized = mname.replace(":", "").replace(" ", "_").replace("=", "").replace("/", "_")
    plt.savefig(out_dir / f"per_regime_diag_{sanitized}.png", dpi=120)
    plt.close()


def plot_yearly_performance_linechart(yearly_df, out_dir):
    piv = yearly_df.pivot_table(index="Year", columns="Model Name", values="R2")
    plt.figure(figsize=(10, 6))
    for col in piv.columns:
        plt.plot(piv.index, piv[col], marker="o", label=col)
    plt.xlabel("Year")
    plt.ylabel("Test R2")
    plt.title("Yearly R2 Stability across Models")
    plt.legend(bbox_to_anchor=(1.05, 1), loc="upper left", fontsize=8)
    plt.grid(True, linestyle="--", alpha=0.5)
    plt.tight_layout()
    plt.savefig(out_dir / "yearly_r2_linechart.png", dpi=150)
    plt.close()

# Section 5: Model matrix and parallel evaluation

Define the model matrix including baseline models (Global V0, Global c1) and 2-regime Mixture-of-Experts (MoE) models across feature arms (`global_v0`, `global_c1`, `spec_new`, `spec_old`) for all 5 gating strategies: `trained_gating_k2`, `Univariate_G_API_k2`, `Clustering_Dynamic_k2`, `Seasonal_Binary_k2`, and `Clustering_V0_Full_k2`. Train all models in parallel using `ThreadPoolExecutor` with CUDA acceleration.

In [5]:
STRATEGIES = [
    "trained_gating_k2",
    "Univariate_G_API_k2",
    "Clustering_Dynamic_k2",
    "Seasonal_Binary_k2",
    "Clustering_V0_Full_k2",
]
ARMS = ["spec_old", "spec_new", "global_v0"]
ARM_LABEL = {
    "spec_old": "Spec-old",
    "spec_new": "Spec-new",
    "global_v0": "Global-V0",
}
STRAT_LABEL = {
    "trained_gating_k2": "Trained Gating K=2",
    "Univariate_G_API_k2": "Univariate G_API K=2",
    "Clustering_Dynamic_k2": "Clustering Dynamic K=2",
    "Seasonal_Binary_k2": "Seasonal Binary K=2",
    "Clustering_V0_Full_k2": "Clustering V0 Full K=2",
}

MODELS_CONFIG = [
    {"id": 1, "name": "Model 1: Baseline V0", "type": "baseline", "arm": "global_v0", "strat": None},
]
mid = 2
for strat in STRATEGIES:
    for arm in ARMS:
        MODELS_CONFIG.append({
            "id": mid,
            "name": f"Model {mid}: {STRAT_LABEL[strat]} ({ARM_LABEL[arm]})",
            "type": "moe",
            "strat": strat,
            "arm": arm,
            "K": 2,
        })
        mid += 1

print(f"Model matrix initialized with {len(MODELS_CONFIG)} models:")
for c in MODELS_CONFIG:
    print(f"  {c['id']:2d}. {c['name']}")


def resolve_specialist_features(strat, arm, cluster_id):
    if arm == "global_v0":
        return list(FEATURE_SET_V0)

    if strat == "trained_gating_k2":
        regime = "dry" if cluster_id == 0 else "wet"
        src = REGIME_OLD if arm == "spec_old" else REGIME_NEW
        return list(src[regime])

    src = CLUSTER_OLD if arm == "spec_old" else CLUSTER_NEW
    key = str(cluster_id)
    if strat not in src or key not in src[strat]:
        return list(FEATURE_SET_V0)
    return list(src[strat][key])


def get_route_labels(strat, K=2):
    if strat == "trained_gating_k2":
        y_gate_trval = np.where(y_trval < T_BINARY, 0, 1)
        y_gate_te_true = np.where(y_te < T_BINARY, 0, 1)
        return {
            "mode": "learned",
            "y_gate_trval": y_gate_trval,
            "y_gate_te_true": y_gate_te_true,
        }

    if strat == "Univariate_G_API_k2":
        binner = QuantileBinner(K)
        binner.fit(train_df["G_API"])
        return {
            "mode": "fixed",
            "labels_trval": binner.predict(trainval_df["G_API"]),
            "labels_te": binner.predict(test_df["G_API"]),
        }

    if strat == "Clustering_Dynamic_k2":
        clusterer = KMeansClusterer(COLS_DYNAMIC, K)
        clusterer.fit(train_df)
        return {
            "mode": "fixed",
            "labels_trval": clusterer.predict(trainval_df),
            "labels_te": clusterer.predict(test_df),
        }

    if strat == "Seasonal_Binary_k2":
        cond_trval = [
            trainval_df["month"].isin([5, 6, 7, 8, 9, 10]),
            trainval_df["month"].isin([11, 12, 1, 2, 3, 4]),
        ]
        cond_te = [
            test_df["month"].isin([5, 6, 7, 8, 9, 10]),
            test_df["month"].isin([11, 12, 1, 2, 3, 4]),
        ]
        return {
            "mode": "fixed",
            "labels_trval": np.select(cond_trval, [0, 1], default=0),
            "labels_te": np.select(cond_te, [0, 1], default=0),
        }

    if strat == "Clustering_V0_Full_k2":
        clusterer = KMeansClusterer(FEATURE_SET_V0, K)
        clusterer.fit(train_df)
        return {
            "mode": "fixed",
            "labels_trval": clusterer.predict(trainval_df),
            "labels_te": clusterer.predict(test_df),
        }

    raise ValueError(f"Unknown strat: {strat}")


models_dir = out_dir / "models"
models_dir.mkdir(parents=True, exist_ok=True)
_save_lock = threading.Lock()
_print_lock = threading.Lock()


def config_crc32(config: dict) -> str:
    payload = {
        "id": config["id"],
        "name": config["name"],
        "type": config["type"],
        "arm": config.get("arm"),
        "strat": config.get("strat"),
        "K": config.get("K", 1),
        "reg_params": XGB_REG_PARAMS,
        "clf_params": XGB_CLF_PARAMS,
    }
    if config["type"] == "moe":
        payload["specialist_features"] = {
            str(c): sorted(resolve_specialist_features(config["strat"], config["arm"], c))
            for c in range(config["K"])
        }
    blob = json.dumps(payload, sort_keys=True).encode("utf-8")
    return f"{zlib.crc32(blob):08x}"


def model_paths(config):
    crc = config_crc32(config)
    mid = config["id"]
    return {
        "crc": crc,
        "pred_cache": models_dir / f"model_{mid}_{crc}_preds.npy",
        "label_cache": models_dir / f"model_{mid}_{crc}_labels_te.npy",
        "curve_cache": models_dir / f"model_{mid}_{crc}_curve.npy",
        "meta_path": models_dir / f"model_{mid}_{crc}_meta.json",
    }


def save_booster(model, path: Path):
    tmp = path.parent / f".{path.name}.tmp"
    model.save_model(tmp)
    tmp.replace(path)


def combine_rmse_curves(spec_rmse_curves, spec_N_tests, n_steps=2500):
    total_N = sum(spec_N_tests)
    if total_N == 0:
        return [0.0] * n_steps
    combined = np.zeros(n_steps, dtype=float)
    for curve, N_c in zip(spec_rmse_curves, spec_N_tests):
        if len(curve) != n_steps:
            full = np.zeros(n_steps, dtype=float)
            full[: len(curve)] = curve
            if len(curve) > 0:
                full[len(curve) :] = curve[-1]
            curve = full
        combined += (N_c / total_N) * (np.asarray(curve) ** 2)
    return np.sqrt(combined).tolist()


def train_one(config):
    paths = model_paths(config)
    crc = paths["crc"]
    model_id = config["id"]
    model_name = config["name"]
    pred_cache = paths["pred_cache"]
    label_cache = paths["label_cache"]
    curve_cache = paths["curve_cache"]
    meta_path = paths["meta_path"]

    if pred_cache.exists() and curve_cache.exists() and meta_path.exists():
        try:
            with open(meta_path, "r") as f:
                meta = json.load(f)
            preds = np.load(pred_cache)
            curve = np.load(curve_cache).tolist()
            labels_te = np.load(label_cache) if label_cache.exists() else None
            with _print_lock:
                print(f"[{model_id}] Loading {model_name} (crc={crc}) from cache...")
            return model_id, {
                "preds": preds,
                "labels_te": labels_te,
                "rmse_curve": curve,
                "train_time_s": meta["train_time_s"],
                "gating": meta.get("gating"),
                "name": meta["name"],
                "crc": crc,
                "loaded": True,
            }
        except Exception as e:
            with _print_lock:
                print(f"[{model_id}] Cache read failed ({e}), re-training...")

    with _print_lock:
        print(f"[{model_id}] Training {model_name} (crc={crc})...")

    t0 = time.perf_counter()
    pred_test = np.zeros(len(test_df), dtype=float)
    gating_info = None
    labels_te = None

    if config["type"] == "baseline":
        feats = (
            FEATURE_SET_V0
            if config["arm"] == "global_v0"
            else FEATURE_SET_C1
        )
        model = XGBRegressor(**XGB_REG_PARAMS)
        model.fit(
            trainval_df[feats],
            y_trval,
            eval_set=[(test_df[feats], y_te)],
            verbose=False,
        )
        pred_test = np.asarray(model.predict(test_df[feats])).ravel()
        rmse_curve = list(model.evals_result()["validation_0"]["rmse"])
        with _save_lock:
            save_booster(model, models_dir / f"model_{model_id}_{crc}_reg.json")

    else:
        strat = config["strat"]
        arm = config["arm"]
        K = config["K"]
        route = get_route_labels(strat, K)

        if route["mode"] == "learned":
            y_gate_trval = route["y_gate_trval"]
            y_gate_te_true = route["y_gate_te_true"]
            gate_feats = FEATURE_SET_V0
            gating_clf = XGBClassifier(**XGB_CLF_PARAMS)
            gating_clf.fit(trainval_df[gate_feats], y_gate_trval, verbose=False)
            pred_gate_te = np.asarray(gating_clf.predict(test_df[gate_feats])).ravel()
            labels_trval = y_gate_trval
            labels_te = pred_gate_te

            acc = accuracy_score(y_gate_te_true, pred_gate_te)
            prec, rec, f1, _ = precision_recall_fscore_support(
                y_gate_te_true, pred_gate_te, average="macro", zero_division=0
            )
            gating_info = {
                "Accuracy": float(acc),
                "Precision": float(prec),
                "Recall": float(rec),
                "F1": float(f1),
                "K": K,
            }
            with _save_lock:
                save_booster(
                    gating_clf,
                    models_dir / f"model_{model_id}_{crc}_gating.json",
                )
        else:
            labels_trval = np.asarray(route["labels_trval"]).ravel()
            labels_te = np.asarray(route["labels_te"]).ravel()

        spec_rmse_curves = []
        spec_N_tests = []
        for c in range(K):
            feats = resolve_specialist_features(strat, arm, c)
            mask_trval = labels_trval == c
            mask_te_sub = labels_te == c
            X_trval_sub = trainval_df.loc[mask_trval, feats]
            y_trval_sub = y_trval[mask_trval]
            X_te_sub = test_df.loc[mask_te_sub, feats]
            y_te_sub = y_te[mask_te_sub]

            specialist = XGBRegressor(**XGB_REG_PARAMS)
            if len(X_te_sub) > 0 and len(X_trval_sub) > 0:
                specialist.fit(
                    X_trval_sub,
                    y_trval_sub,
                    eval_set=[(X_te_sub, y_te_sub)],
                    verbose=False,
                )
                curve = list(specialist.evals_result()["validation_0"]["rmse"])
            elif len(X_trval_sub) > 0:
                specialist.fit(X_trval_sub, y_trval_sub, verbose=False)
                curve = [0.0] * 2500
            else:
                curve = [0.0] * 2500
                if mask_te_sub.any():
                    pred_test[mask_te_sub] = float(np.mean(y_trval))
                spec_rmse_curves.append(curve)
                spec_N_tests.append(int(mask_te_sub.sum()))
                continue

            with _save_lock:
                save_booster(
                    specialist,
                    models_dir / f"model_{model_id}_{crc}_spec{c}.json",
                )
            if mask_te_sub.any():
                pred_test[mask_te_sub] = np.asarray(
                    specialist.predict(X_te_sub)
                ).ravel()
            spec_rmse_curves.append(curve)
            spec_N_tests.append(int(mask_te_sub.sum()))

        rmse_curve = combine_rmse_curves(spec_rmse_curves, spec_N_tests)

    train_time = time.perf_counter() - t0
    meta = {
        "train_time_s": train_time,
        "name": model_name,
        "config_crc32": crc,
        "config_id": model_id,
        "gating": gating_info,
        "arm": config.get("arm"),
        "strat": config.get("strat"),
        "type": config["type"],
    }
    with _save_lock:
        np.save(pred_cache, pred_test)
        if labels_te is not None:
            np.save(label_cache, labels_te)
        np.save(curve_cache, np.asarray(rmse_curve, dtype=float))
        tmp_meta = meta_path.parent / f".{meta_path.stem}.writing.json"
        with open(tmp_meta, "w") as f:
            json.dump(meta, f, indent=2)
        tmp_meta.replace(meta_path)

    with _print_lock:
        print(f"[{model_id}] Done in {train_time:.2f}s (crc={crc})")

    return model_id, {
        "preds": pred_test,
        "labels_te": labels_te,
        "rmse_curve": rmse_curve,
        "train_time_s": train_time,
        "gating": gating_info,
        "name": model_name,
        "crc": crc,
        "loaded": False,
    }


print(f"\nStarting parallel training for {len(MODELS_CONFIG)} models "
      f"with {PARALLEL_WORKERS} workers...\n")
wall_t0 = time.perf_counter()
results_by_id = {}
failed = []
completed = 0

with ThreadPoolExecutor(max_workers=PARALLEL_WORKERS) as executor:
    futures = {executor.submit(train_one, cfg): cfg for cfg in MODELS_CONFIG}
    for fut in as_completed(futures):
        cfg = futures[fut]
        mid = cfg["id"]
        try:
            mid, result = fut.result()
            results_by_id[mid] = result
        except Exception as e:
            failed.append({
                "id": mid,
                "name": cfg.get("name"),
                "error": f"{type(e).__name__}: {e}",
            })
            with _print_lock:
                print(f"[{mid}] FAILED: {type(e).__name__}: {e}")
        completed += 1
        if completed % 3 == 0 or completed == len(MODELS_CONFIG):
            elapsed = time.perf_counter() - wall_t0
            rate = completed / elapsed if elapsed > 0 else 0
            remaining = (len(MODELS_CONFIG) - completed) / rate if rate > 0 else float("nan")
            n_loaded = sum(1 for r in results_by_id.values() if r.get("loaded"))
            with _print_lock:
                print(
                    f"  Progress: {completed}/{len(MODELS_CONFIG)} "
                    f"({elapsed:.1f}s elapsed, ~{remaining:.1f}s remaining, "
                    f"{len(failed)} failed, {n_loaded} loaded from cache)"
                )

wall_t1 = time.perf_counter()
print(
    f"\nProcessed {completed} configs in {wall_t1 - wall_t0:.1f}s wall time "
    f"({len(results_by_id)} ok, {len(failed)} failed)."
)

overall_metrics = []
per_regime_metrics = []
yearly_metrics = []
gating_perf_metrics = []
all_rmse_curves = {}

for config in MODELS_CONFIG:
    mid = config["id"]
    mname = config["name"]
    if mid not in results_by_id:
        print(f"[SKIP] No results for {mname}")
        continue
    result = results_by_id[mid]
    pred_test = result["preds"]
    labels_te = result.get("labels_te")
    all_rmse_curves[mname] = result["rmse_curve"]

    metrics_ov = compute_metrics(y_te, pred_test)
    metrics_ov["Model Name"] = mname
    metrics_ov["Model ID"] = mid
    metrics_ov["Arm"] = config.get("arm")
    metrics_ov["Strategy"] = config.get("strat")
    metrics_ov["Train Time (s)"] = result["train_time_s"]
    overall_metrics.append(metrics_ov)
    print(f"{mname}: R2 = {metrics_ov['R2']:.4f}")

    if labels_te is not None:
        mask0 = labels_te == 0
        mask1 = labels_te == 1
        m_r0 = compute_metrics(y_te[mask0], pred_test[mask0]) if mask0.any() else {}
        m_r1 = compute_metrics(y_te[mask1], pred_test[mask1]) if mask1.any() else {}
        per_regime_metrics.append({
            "Model ID": mid,
            "Model Name": mname,
            "Strategy": config.get("strat"),
            "Arm": config.get("arm"),
            "N_R0": int(mask0.sum()),
            "R2_R0": m_r0.get("R2", float("nan")),
            "RMSE_R0": m_r0.get("RMSE", float("nan")),
            "Bias_R0": m_r0.get("Bias", float("nan")),
            "MAE_R0": m_r0.get("MAE", float("nan")),
            "N_R1": int(mask1.sum()),
            "R2_R1": m_r1.get("R2", float("nan")),
            "RMSE_R1": m_r1.get("RMSE", float("nan")),
            "Bias_R1": m_r1.get("Bias", float("nan")),
            "MAE_R1": m_r1.get("MAE", float("nan")),
        })

    for yr in test_years:
        mask_yr = (test_df["year"] == yr).values
        metrics_yr = compute_metrics(y_te[mask_yr], pred_test[mask_yr])
        metrics_yr["Model Name"] = mname
        metrics_yr["Model ID"] = mid
        metrics_yr["Year"] = int(yr)
        yearly_metrics.append(metrics_yr)

    if result.get("gating"):
        g = dict(result["gating"])
        g["Model Name"] = mname
        gating_perf_metrics.append(g)

    plot_diagnostics(mname, y_te, pred_test, test_df, out_dir)
    if labels_te is not None:
        plot_per_regime_diagnostics(mname, y_te, pred_test, labels_te, test_df, out_dir)

rmse_curves_df = pd.DataFrame(all_rmse_curves)
rmse_curves_df.index.name = "Step"
rmse_curves_df.to_csv(out_dir / "all_models_loss_curves.csv")
print(f"Saved loss curves to {out_dir / 'all_models_loss_curves.csv'}")

if failed:
    raise RuntimeError(
        f"{len(failed)} configs failed (see failed_configs.csv). "
        f"{len(results_by_id)} succeeded — re-run to resume."
    )

Model matrix initialized with 16 models:
   1. Model 1: Baseline V0
   2. Model 2: Trained Gating K=2 (Spec-old)
   3. Model 3: Trained Gating K=2 (Spec-new)
   4. Model 4: Trained Gating K=2 (Global-V0)
   5. Model 5: Univariate G_API K=2 (Spec-old)
   6. Model 6: Univariate G_API K=2 (Spec-new)
   7. Model 7: Univariate G_API K=2 (Global-V0)
   8. Model 8: Clustering Dynamic K=2 (Spec-old)
   9. Model 9: Clustering Dynamic K=2 (Spec-new)
  10. Model 10: Clustering Dynamic K=2 (Global-V0)
  11. Model 11: Seasonal Binary K=2 (Spec-old)
  12. Model 12: Seasonal Binary K=2 (Spec-new)
  13. Model 13: Seasonal Binary K=2 (Global-V0)
  14. Model 14: Clustering V0 Full K=2 (Spec-old)
  15. Model 15: Clustering V0 Full K=2 (Spec-new)
  16. Model 16: Clustering V0 Full K=2 (Global-V0)

Starting parallel training for 16 models with 4 workers...

[2] Training Model 2: Trained Gating K=2 (Spec-old) (crc=9cbba4c4)...
[3] Training Model 3: Trained Gating K=2 (Spec-new) (crc=7c320593)...
[4] Trainin

[1] Loading Model 1: Baseline V0 (crc=1f42ac84) from cache...
[5] Training Model 5: Univariate G_API K=2 (Spec-old) (crc=fb0501bc)...


[22:52:10] WARNING: /__w/xgboost/xgboost/src/common/error_msg.cc:62: Falling back to prediction using DMatrix due to mismatched devices. This might lead to higher memory usage and slower performance. XGBoost is running on: cuda:0, while the input data is on: cpu.
Potential solutions:
- Use a data structure that matches the device ordinal in the booster.
- Set the device for booster before call to inplace_predict.

This warning will only be shown once.

[22:52:10] WARNING: /__w/xgboost/xgboost/src/c_api/c_api.cc:1573: Saving model in the UBJSON format as default.  You can use a file extension: `json` or `ubj` to choose between formats.
[22:52:10] WARNING: /__w/xgboost/xgboost/src/c_api/c_api.cc:1573: Saving model in the UBJSON format as default.  You can use a file extension: `json` or `ubj` to choose between formats.
[22:52:10] WARNING: /__w/xgboost/xgboost/src/c_api/c_api.cc:1573: Saving model in the UBJSON format as default.  You can use a file extension: `json` or `ubj` to choose be

[22:52:23] WARNING: /__w/xgboost/xgboost/src/c_api/c_api.cc:1573: Saving model in the UBJSON format as default.  You can use a file extension: `json` or `ubj` to choose between formats.


[22:52:25] WARNING: /__w/xgboost/xgboost/src/c_api/c_api.cc:1573: Saving model in the UBJSON format as default.  You can use a file extension: `json` or `ubj` to choose between formats.
[22:52:25] WARNING: /__w/xgboost/xgboost/src/c_api/c_api.cc:1573: Saving model in the UBJSON format as default.  You can use a file extension: `json` or `ubj` to choose between formats.


...

Model 13: Seasonal Binary K=2 (Global-V0): R2 = 0.6165


Model 14: Clustering V0 Full K=2 (Spec-old): R2 = 0.6619


Model 15: Clustering V0 Full K=2 (Spec-new): R2 = 0.5212


Model 16: Clustering V0 Full K=2 (Global-V0): R2 = 0.6619


Saved loss curves to /scratch/user/u.rp352032/MDR-Project/notebooks/experiment/derived_8.3-eval-1.0/all_models_loss_curves.csv


# Section 6: Results tables, figures, and export

Consolidate metrics across all models, display sorted leaderboard, print per-regime breakdown and yearly performance tables, render consolidated and strategy-grouped loss curves, plot per-regime $R^2$/RMSE comparisons and residual boxplots, and save summary CSV artifacts.

In [6]:
overall_metrics_df = pd.DataFrame(overall_metrics).sort_values("R2", ascending=False)
print("=== OVERALL METRICS (sorted by R2) ===")
print(
    overall_metrics_df[
        ["Model ID", "Model Name", "Arm", "Strategy", "R2", "RMSE", "ubRMSE", "Bias", "MAE", "Med|Err|", "Pearson", "Train Time (s)"]
    ].to_string(
        index=False,
        formatters={
            "R2": "{:,.4f}".format,
            "RMSE": "{:,.4f}".format,
            "ubRMSE": "{:,.4f}".format,
            "Bias": "{:+,.4f}".format,
            "MAE": "{:,.4f}".format,
            "Med|Err|": "{:,.4f}".format,
            "Pearson": "{:,.4f}".format,
            "Train Time (s)": "{:,.1f}".format,
        },
    )
)
overall_metrics_df.to_csv(out_dir / "metrics_summary.csv", index=False)
print(f"\nSaved overall metrics to: {out_dir / 'metrics_summary.csv'}")

if per_regime_metrics:
    per_regime_df = pd.DataFrame(per_regime_metrics)
    per_regime_df.to_csv(out_dir / "per_regime_metrics_summary.csv", index=False)
    print(f"Saved per-regime metrics summary to: {out_dir / 'per_regime_metrics_summary.csv'}")
    print("\n=== PER-REGIME METRICS SUMMARY ===")
    print(
        per_regime_df[
            ["Model ID", "Model Name", "N_R0", "R2_R0", "RMSE_R0", "Bias_R0", "N_R1", "R2_R1", "RMSE_R1", "Bias_R1"]
        ].to_string(
            index=False,
            formatters={
                "R2_R0": "{:,.4f}".format,
                "RMSE_R0": "{:,.4f}".format,
                "Bias_R0": "{:+,.4f}".format,
                "R2_R1": "{:,.4f}".format,
                "RMSE_R1": "{:,.4f}".format,
                "Bias_R1": "{:+,.4f}".format,
            },
        )
    )

yearly_metrics_df = pd.DataFrame(yearly_metrics)
yearly_metrics_df.to_csv(out_dir / "metrics_by_year.csv", index=False)
print(f"Saved yearly metrics to: {out_dir / 'metrics_by_year.csv'}")

pivot = yearly_metrics_df.pivot_table(index="Model Name", columns="Year", values="R2")
overall_r2 = overall_metrics_df.set_index("Model Name")["R2"]
pivot.insert(0, "Overall", overall_r2)
print("\n=== R2 BY YEAR ===")
print(pivot.round(4).to_string())

plot_yearly_performance_linechart(yearly_metrics_df, out_dir)

if len(gating_perf_metrics) > 0:
    gating_perf_df = pd.DataFrame(gating_perf_metrics)
    print("\n=== GATING ROUTER PERFORMANCE (trained gating only) ===")
    print(gating_perf_df.to_string(index=False))
    gating_perf_df.to_csv(out_dir / "gating_performance_summary.csv", index=False)

curves_df = rmse_curves_df
plt.figure(figsize=(14, 8))
colors = cm.tab20(np.linspace(0, 1, len(curves_df.columns)))
for idx, col in enumerate(curves_df.columns):
    plt.plot(curves_df.index, curves_df[col], label=col, color=colors[idx], linewidth=1.2)
plt.xlabel("Training Step (Boosting Iteration)", fontweight="bold")
plt.ylabel("Test RMSE", fontweight="bold")
plt.title("Consolidated Loss Curves (RMSE) on Test Set (derived_8.3)", fontsize=14, fontweight="bold")
plt.legend(bbox_to_anchor=(1.05, 1), loc="upper left", fontsize=7)
plt.grid(True, linestyle="--", alpha=0.6)
plt.tight_layout()
plt.savefig(out_dir / "loss_curves_consolidated.png", dpi=150)
plt.close()

fig, axes = plt.subplots(3, 2, figsize=(16, 15))
axes = axes.ravel()
arm_styles = {
    "spec_old": ("#1f77b4", "-"),
    "spec_new": ("#2ca02c", "-"),
    "global_v0": ("#d62728", "--"),
}
baseline_v0_name = "Model 1: Baseline V0"

for idx, strat in enumerate(STRATEGIES):
    ax = axes[idx]
    if baseline_v0_name in curves_df.columns:
        ax.plot(curves_df.index, curves_df[baseline_v0_name], "k-", alpha=0.5, label="Baseline V0", linewidth=1.5)
    for arm in ARMS:
        matches = [
            c["name"]
            for c in MODELS_CONFIG
            if c.get("strat") == strat and c.get("arm") == arm
        ]
        if not matches or matches[0] not in curves_df.columns:
            continue
        color, ls = arm_styles[arm]
        ax.plot(
            curves_df.index,
            curves_df[matches[0]],
            color=color,
            linestyle=ls,
            label=ARM_LABEL[arm],
            linewidth=2,
        )
    ax.set_title(STRAT_LABEL[strat], fontsize=12, fontweight="bold")
    ax.set_xlabel("Training Step")
    ax.set_ylabel("Test RMSE")
    ax.legend(fontsize=8)
    ax.grid(True, linestyle="--", alpha=0.5)

if len(STRATEGIES) < len(axes):
    axes[-1].axis("off")

plt.suptitle(
    "Loss Curves by Strategy: Spec-old vs Spec-new vs Global-V0 vs Global-c1 (derived_8.3)",
    fontsize=14,
    fontweight="bold",
    y=0.995,
)
plt.tight_layout()
plt.savefig(out_dir / "loss_curves_grouped.png", dpi=150)
plt.close()

if per_regime_metrics:
    pr_df = pd.DataFrame(per_regime_metrics)
    fig, axes = plt.subplots(1, 2, figsize=(14, 6))

    ax = axes[0]
    x = np.arange(len(pr_df))
    width = 0.35
    ax.bar(x - width/2, pr_df["R2_R0"], width, label="Regime 0 (Dry)", color="#1f77b4")
    ax.bar(x + width/2, pr_df["R2_R1"], width, label="Regime 1 (Wet)", color="#ff7f0e")
    ax.set_xticks(x)
    ax.set_xticklabels(pr_df["Model ID"], rotation=0)
    ax.set_xlabel("Model ID")
    ax.set_ylabel("R2 Score")
    ax.set_title("Per-Regime R2 Comparison (Regime 0 vs Regime 1)", fontweight="bold")
    ax.axhline(0, color="k", linestyle="--", linewidth=0.8)
    ax.legend()
    ax.grid(True, linestyle="--", alpha=0.5)

    ax = axes[1]
    ax.bar(x - width/2, pr_df["RMSE_R0"], width, label="Regime 0 (Dry)", color="#2ca02c")
    ax.bar(x + width/2, pr_df["RMSE_R1"], width, label="Regime 1 (Wet)", color="#d62728")
    ax.set_xticks(x)
    ax.set_xticklabels(pr_df["Model ID"], rotation=0)
    ax.set_xlabel("Model ID")
    ax.set_ylabel("RMSE")
    ax.set_title("Per-Regime RMSE Comparison (Regime 0 vs Regime 1)", fontweight="bold")
    ax.legend()
    ax.grid(True, linestyle="--", alpha=0.5)

    plt.tight_layout()
    plt.savefig(out_dir / "per_regime_r2_rmse_comparison.png", dpi=150)
    plt.close()

if per_regime_metrics:
    box_data = []
    labels_list = []
    for cfg in MODELS_CONFIG:
        mid = cfg["id"]
        if mid not in results_by_id:
            continue
        res_info = results_by_id[mid]
        labels_te = res_info.get("labels_te")
        if labels_te is None:
            continue
        preds = res_info["preds"]
        resids = preds - y_te
        for c in range(2):
            mask = labels_te == c
            if mask.any():
                box_data.append(resids[mask])
                labels_list.append(f"M{mid}_R{c}")

    if box_data:
        plt.figure(figsize=(16, 6))
        plt.boxplot(box_data, tick_labels=labels_list, patch_artist=True,
                    boxprops=dict(facecolor="#cbd5e1", color="#334155"),
                    medianprops=dict(color="#d62728", linewidth=1.5))
        plt.axhline(0, color="k", linestyle="--", alpha=0.7)
        plt.ylabel("Residual (Pred - Obs)", fontweight="bold")
        plt.title("Per-Regime Residual Distributions across MoE Models", fontweight="bold")
        plt.xticks(rotation=90, fontsize=8)
        plt.grid(True, linestyle="--", alpha=0.5)
        plt.tight_layout()
        plt.savefig(out_dir / "per_regime_residuals_boxplot.png", dpi=150)
        plt.close()

ablation_rows = []
for c in MODELS_CONFIG:
    if c["type"] != "moe":
        continue
    row = overall_metrics_df[overall_metrics_df["Model ID"] == c["id"]]
    if row.empty:
        continue
    ablation_rows.append({
        "Strategy": STRAT_LABEL[c["strat"]],
        "Arm": ARM_LABEL[c["arm"]],
        "R2": float(row["R2"].iloc[0]),
        "RMSE": float(row["RMSE"].iloc[0]),
    })
ablation_df = pd.DataFrame(ablation_rows)
if not ablation_df.empty:
    pivot_abl = ablation_df.pivot(index="Strategy", columns="Arm", values="R2")
    pivot_abl.to_csv(out_dir / "ablation_r2_strategy_x_arm.csv")
    print("\n=== ABLATION TABLE: R2 BY STRATEGY x ARM ===")
    print(pivot_abl.round(4).to_string())

print("\nAll evaluation analyses, charts, and metric tables generated successfully.")

=== OVERALL METRICS (sorted by R2) ===
 Model ID                                   Model Name       Arm              Strategy     R2   RMSE ubRMSE    Bias    MAE Med|Err| Pearson Train Time (s)
       14  Model 14: Clustering V0 Full K=2 (Spec-old)  spec_old Clustering_V0_Full_k2 0.6619 0.0604 0.0584 +0.0155 0.0435   0.0315  0.8282           29.0
       16 Model 16: Clustering V0 Full K=2 (Global-V0) global_v0 Clustering_V0_Full_k2 0.6619 0.0604 0.0584 +0.0155 0.0435   0.0315  0.8282           28.0
        1                         Model 1: Baseline V0 global_v0                   NaN 0.6435 0.0620 0.0601 +0.0155 0.0453   0.0336  0.8172           21.7
       10 Model 10: Clustering Dynamic K=2 (Global-V0) global_v0 Clustering_Dynamic_k2 0.6243 0.0637 0.0613 +0.0173 0.0466   0.0346  0.8100           35.0
       13    Model 13: Seasonal Binary K=2 (Global-V0) global_v0    Seasonal_Binary_k2 0.6165 0.0644 0.0629 +0.0136 0.0469   0.0352  0.7984           34.9
        7    Model 7: Univariat


=== GATING ROUTER PERFORMANCE (trained gating only) ===
 Accuracy  Precision   Recall       F1  K                              Model Name
  0.87768   0.870164 0.863254 0.866467  2  Model 2: Trained Gating K=2 (Spec-old)
  0.87768   0.870164 0.863254 0.866467  2  Model 3: Trained Gating K=2 (Spec-new)
  0.87768   0.870164 0.863254 0.866467  2 Model 4: Trained Gating K=2 (Global-V0)



=== ABLATION TABLE: R2 BY STRATEGY x ARM ===
Arm                     Global-V0  Spec-new  Spec-old
Strategy                                             
Clustering Dynamic K=2     0.6243    0.5445    0.5660
Clustering V0 Full K=2     0.6619    0.5212    0.6619
Seasonal Binary K=2        0.6165    0.5261    0.5662
Trained Gating K=2         0.5863    0.5272    0.5595
Univariate G_API K=2       0.6137    0.5605    0.5086

All evaluation analyses, charts, and metric tables generated successfully.
